# HayFlow — real Hay L5PC teacher manifest

This notebook runs the first HayFlow teacher milestone in a Linux notebook environment (Kaggle, Colab, or Jupyter/HPC). It installs a pinned NEURON runtime, checks out the exact teacher commit, compiles the original `.mod` mechanisms, builds the full instantiated morphology manifest, and validates the result.

The structural pass intentionally contains no synapses yet: the original generator creates those point processes inside each simulation. A later instrumentation hook will append them after instantiation.

Internet access is required only when the repositories or Python wheel are not already available in the notebook environment.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

NEURON_VERSION = "8.2.7"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    f"neuron=={NEURON_VERSION}",
])

import neuron

print("Python:", sys.version)
print("NEURON:", neuron.__version__)
assert neuron.__version__ == NEURON_VERSION


## Workspace and pinned sources

Set `HAYFLOW_ELM_REPO` when the current `elmneuron` checkout is mounted as a notebook dataset. Otherwise the notebook clones `Zagred47/giada`. The teacher is always checked against its exact source commit.

In [ ]:
ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
TEACHER_REPOSITORY = (
    "https://github.com/SelfishGene/neuron_as_deep_net.git"
)
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"

if Path("/kaggle/working").is_dir():
    NOTEBOOK_ROOT = Path("/kaggle/working")
elif Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content")
else:
    NOTEBOOK_ROOT = Path.cwd()

WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
ELM_REPO = (
    Path(elm_override).expanduser().resolve()
    if elm_override
    else WORKSPACE / "elmneuron"
)
TEACHER_REPO = WORKSPACE / "neuron_as_deep_net"
MANIFEST_PATH = (
    WORKSPACE / "artifacts" / "teacher_manifest.json"
)

print("Workspace:", WORKSPACE)
print("ELM repo:", ELM_REPO)
print("Teacher repo:", TEACHER_REPO)


In [ ]:
def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(
        list(map(str, command)), cwd=cwd, check=True
    )


if not (ELM_REPO / ".git").is_dir():
    if elm_override:
        raise FileNotFoundError(
            f"HAYFLOW_ELM_REPO is not a Git checkout: {ELM_REPO}"
        )
    run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "checkout", ELM_REF], cwd=ELM_REPO)

if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)

teacher_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True
).strip()
assert teacher_head == TEACHER_COMMIT, (teacher_head, TEACHER_COMMIT)
assert (ELM_REPO / "src" / "hayflow_teacher").is_dir(), (
    "The selected elmneuron checkout does not contain HayFlow. "
    "Mount the current checkout or select the branch containing it."
)
print("Pinned teacher commit:", teacher_head)


## Compile the original mechanisms

The command runs from `L5PC_NEURON_simulation`, so `neuron.load_mechanisms` can later discover the generated architecture directory. No `.mod` source is changed.

In [ ]:
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
MOD_DIR = SIMULATION_DIR / "mods"
assert MOD_DIR.is_dir()

nrnivmodl = shutil.which("nrnivmodl")
if nrnivmodl is None:
    candidate = Path(sys.executable).parent / "nrnivmodl"
    nrnivmodl = str(candidate) if candidate.is_file() else None
if nrnivmodl is None:
    raise RuntimeError("nrnivmodl was not installed with NEURON")
if shutil.which("gcc") is None:
    raise RuntimeError("A C compiler is required to build the .mod files")

run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
compiled_candidates = [
    SIMULATION_DIR / arch / relative
    for arch in ("x86_64", "aarch64")
    for relative in ("special", "libnrnmech.so", ".libs/libnrnmech.so")
]
compiled_artifact = next(
    (path for path in compiled_candidates if path.is_file()),
    None,
)
assert compiled_artifact is not None, (
    "nrnivmodl completed but no compiled mechanism artifact was found"
)


## Run unit tests before touching the teacher

In [ ]:
run([
    sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_*.py", "-v",
], cwd=ELM_REPO)


## Instantiate Hay L5PC and build the manifest

In [ ]:
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
run([
    sys.executable,
    ELM_REPO / "scripts" / "hayflow" / "build_teacher_manifest.py",
    "--teacher-repo", TEACHER_REPO,
    "--output", MANIFEST_PATH,
], cwd=ELM_REPO)
assert MANIFEST_PATH.is_file()
print("Manifest:", MANIFEST_PATH)


## Validate topology and provenance

This cell rejects a dendrite-only manifest, an incorrect commit, a malformed Hines order, or a child appearing before its parent.

In [ ]:
import json
from collections import Counter

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
sections = manifest["sections"]
segments = manifest["segments"]
variables = manifest["variables"]
hines_order = manifest["hines_order"]

assert manifest["source_commit"] == TEACHER_COMMIT
assert manifest["neuron_version"].startswith(NEURON_VERSION)
assert len(sections) > 0
assert len(segments) > 639, "Expected soma/axon in addition to dendrites"
assert len(variables) > len(segments)
assert sorted(hines_order) == list(range(len(segments)))

hines_position = {
    segment_id: position
    for position, segment_id in enumerate(hines_order)
}
for segment in segments:
    parent = segment["parent_segment_id"]
    if parent is not None:
        assert hines_position[parent] < hines_position[segment["id"]]

region_counts = Counter(item["region"] for item in segments)
variable_counts = Counter(item["kind"] for item in variables)
print("Sections:", len(sections))
print("Segments:", len(segments))
print("Variables:", len(variables))
print("Synapses in structural pass:", len(manifest["synapses"]))
print("Regions:", dict(region_counts))
print("Variable kinds:", dict(variable_counts))
print("Metadata:", manifest["metadata"])


## Preserve the artifact

Download or publish `MANIFEST_PATH` together with the notebook output. Do not treat it as the final teacher contract until the synapse hook and validated apical subregion map have been added.

In [ ]:
print(MANIFEST_PATH)
print("Size (KiB):", MANIFEST_PATH.stat().st_size / 1024.0)
print("SHA256:")
import hashlib
print(hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest())
